# Bioresearch — GRPO Training on Virtual Tumor Board

This Colab notebook fine-tunes a small Qwen 2.5 model on the Bioresearch environment using **Group Relative Policy Optimization (GRPO)** via HuggingFace TRL + Unsloth.

**Runtime:** free Colab T4 (~45 min for 150 steps).

**What you get:**

- A trained LoRA adapter on `Qwen/Qwen2.5-1.5B-Instruct`
- Reward curves (mean, std, advantage)
- Before/after evaluation table
- Checkpoint ready to push to Hugging Face Hub

**Pipeline overview:**

1. Install Unsloth + TRL + OpenEnv deps
2. Clone `bioresearch` and import the environment in-process
3. Define a TRL-compatible reward function that wraps `BioresearchEnvironment`
4. Load Qwen 2.5 1.5B with 4-bit quantisation
5. Train with `GRPOTrainer`
6. Plot reward curves and save the adapter

## 1. Install dependencies

Unsloth gives us 2x throughput and native 4-bit. TRL provides `GRPOTrainer`. We also pin `openenv-core` for the environment interface types.

In [ ]:
# Colab only: uncomment to install
# !pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
# !pip install -q --no-deps 'trl>=0.14' peft accelerate bitsandbytes
# !pip install -q 'openenv-core[core]>=0.2.2' pydantic datasets matplotlib

import sys, subprocess, importlib.util

def ensure(pkg):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for p in ['trl', 'transformers', 'accelerate', 'peft', 'datasets', 'matplotlib']:
    ensure(p)

print('Dependencies OK')

## 2. Clone the Bioresearch repo

In [ ]:
import os, subprocess, sys

REPO_DIR = '/content/bioresearch' if os.path.exists('/content') else os.path.abspath('..')

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', 'https://github.com/YOUR_USERNAME/bioresearch.git', REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print('Using repo at', REPO_DIR)

## 3. Import the environment

In [ ]:
from models import BioresearchAction
from server.bioresearch_environment import BioresearchEnvironment
from server.data_loader import DataLoader

env = BioresearchEnvironment()
data = DataLoader()

print(f'DNA samples: {data.dna_count}, Protein samples: {data.protein_count}')
print(f'Diseases in vocabulary: {len(data.all_disease_answers)}')

## 4. Build the prompt dataset

We use `dna_classification` — the fastest grader with the highest per-step variance, which is ideal for early GRPO convergence. Switch `TASK_TYPE` to `virtual_tumor_board` for a second-stage run.

In [ ]:
from datasets import Dataset

TASK_TYPE = 'dna_classification'

train_task_ids = data.get_all_dna_ids(baseline_only=False)  # 80 samples
eval_task_ids = data.get_all_dna_ids(baseline_only=True)    # 20 held-out

SYSTEM_PROMPT = (
    'You are a genomics expert. Given a DNA variant and its pathway context, '
    'identify which disease this mutation contributes to. Reply with ONLY the disease name.'
)

def build_record(task_id: str):
    sample = data.get_dna_sample_by_id(task_id)
    prompt = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': sample.question},
    ]
    return {'task_id': task_id, 'prompt': prompt, 'gold_answer': sample.answer}

train_ds = Dataset.from_list([build_record(tid) for tid in train_task_ids])
eval_ds  = Dataset.from_list([build_record(tid) for tid in eval_task_ids])
print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

## 5. Reward function

The reward function runs the environment in-process. Each completion in the GRPO group is graded against the same `task_id` — guaranteeing the per-group variance that GRPO needs.

In [ ]:
import re

def extract_answer(text: str) -> str:
    t = (text or '').strip()
    # strip leading/trailing quotes, bullets, JSON
    t = re.sub(r'^["\'`\-\*\s]+|["\'`\s]+$', '', t)
    # take first line if multi-line
    return t.split('\n')[0][:120]

def bioresearch_reward(prompts, completions, task_id, gold_answer, **_):
    """GRPO-compatible reward: returns List[float] parallel to completions."""
    rewards = []
    for tid, completion in zip(task_id, completions):
        text = completion if isinstance(completion, str) else completion[0]['content']
        answer = extract_answer(text)
        env.reset(task_type=TASK_TYPE, task_id=tid)
        obs = env.step(BioresearchAction(task_id=tid, answer=answer))
        rewards.append(float(obs.reward or 0.0))
    return rewards

# quick sanity check
sample_row = train_ds[0]
fake_completions = [[{'role': 'assistant', 'content': sample_row['gold_answer']}]]
r = bioresearch_reward(
    prompts=[sample_row['prompt']],
    completions=fake_completions,
    task_id=[sample_row['task_id']],
    gold_answer=[sample_row['gold_answer']],
)
print('Sanity reward (should be ~0.9):', r)

## 6. Load the model with Unsloth

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LENGTH = 2048

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth',
    )
except ImportError:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype='auto')
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=16, target_modules=['q_proj', 'v_proj']))

print('Model loaded.')

## 7. GRPO training

Key hyperparameters:

- `num_generations=4` — GRPO group size
- `learning_rate=5e-6` — low LR for stable policy updates
- `beta=0.04` — KL regularization against the reference policy

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='grpo_bioresearch',
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=64,
    max_steps=150,
    logging_steps=1,
    save_steps=50,
    report_to='none',
    beta=0.04,
    temperature=0.9,
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    processing_class=tokenizer,
    train_dataset=train_ds,
    reward_funcs=[bioresearch_reward],
)

trainer.train()

## 8. Plot the reward curve

In [ ]:
import matplotlib.pyplot as plt

log_hist = trainer.state.log_history
steps = [h.get('step', i) for i, h in enumerate(log_hist) if 'reward' in h]
rewards = [h['reward'] for h in log_hist if 'reward' in h]
reward_stds = [h.get('reward_std', 0.0) for h in log_hist if 'reward' in h]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(steps, rewards, label='mean reward', color='#2563eb', linewidth=2)
if reward_stds and any(s > 0 for s in reward_stds):
    ax.fill_between(
        steps,
        [r - s for r, s in zip(rewards, reward_stds)],
        [r + s for r, s in zip(rewards, reward_stds)],
        alpha=0.2, color='#2563eb',
    )
ax.set_xlabel('Training step')
ax.set_ylabel('Reward')
ax.set_title(f'GRPO training reward curve — {TASK_TYPE}')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()

## 9. Before/after evaluation

We evaluate the fine-tuned model on the held-out `eval_ds` and compare to a pre-training pass.

In [ ]:
import torch

def generate_completion(messages, model, tokenizer, max_new_tokens=64, temperature=0.0):
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            inputs, max_new_tokens=max_new_tokens,
            do_sample=temperature > 0, temperature=max(temperature, 0.01),
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return text

def evaluate(ds, model, tokenizer, n=20):
    rewards = []
    for row in ds.select(range(min(n, len(ds)))):
        completion = generate_completion(row['prompt'], model, tokenizer)
        env.reset(task_type=TASK_TYPE, task_id=row['task_id'])
        obs = env.step(BioresearchAction(task_id=row['task_id'], answer=extract_answer(completion)))
        rewards.append(float(obs.reward or 0.0))
    return sum(rewards) / len(rewards), rewards

after_mean, after_rewards = evaluate(eval_ds, model, tokenizer, n=20)
print(f'Post-training mean reward (n=20): {after_mean:.4f}')

## 10. Save and push to Hugging Face Hub

Replace the repo name with your own. You'll need `huggingface-cli login` first.

In [ ]:
OUTPUT_REPO = 'YOUR_USERNAME/bioresearch-grpo-qwen2.5-1.5b'

trainer.save_model('grpo_bioresearch_final')
# trainer.push_to_hub(OUTPUT_REPO)
print('Saved locally. Uncomment push_to_hub to upload.')